In [ ]:
# Lab type: review
# Course: EDA — Exploratory Data Analysis
# Lesson: Grouping and Segmentation
# Task: Review the code in each section. The code is correct and runs without errors.
#       Your job is to evaluate the analytical choices — and answer the judgment questions
#       in the comment cells. There are no bugs to fix; there are decisions to interrogate.

## Setup

Run this cell first. It installs dependencies and generates a synthetic orders dataset that mirrors the lesson examples.

In [ ]:
!pip install pandas matplotlib numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

n = 85_000

channel_probs = [0.49, 0.34, 0.13, 0.04]
channels = rng.choice(["online", "retail", "wholesale", "enterprise"], size=n, p=channel_probs)

channel_params = {
    "online":     {"loc": 44,  "scale": 31,  "skew": 2.5},
    "retail":     {"loc": 56,  "scale": 42,  "skew": 2.0},
    "wholesale":  {"loc": 153, "scale": 119, "skew": 1.8},
    "enterprise": {"loc": 418, "scale": 387, "skew": 1.2},
}

revenue = np.empty(n)
for ch, params in channel_params.items():
    mask = channels == ch
    raw = rng.lognormal(mean=np.log(params["loc"]), sigma=0.6, size=mask.sum())
    revenue[mask] = np.clip(raw, 5, params["loc"] * 12)

regions = rng.choice(["North", "South", "East", "West", None], size=n,
                     p=[0.27, 0.25, 0.26, 0.20, 0.02])

status_probs = {
    "online":     [0.71, 0.11, 0.09, 0.06, 0.03],
    "retail":     [0.74, 0.10, 0.08, 0.05, 0.03],
    "wholesale":  [0.77, 0.10, 0.07, 0.04, 0.02],
    "enterprise": [0.76, 0.11, 0.08, 0.03, 0.02],
}
statuses = ["delivered", "shipped", "cancelled", "returned", "pending"]
order_status = np.empty(n, dtype=object)
for ch, probs in status_probs.items():
    mask = channels == ch
    order_status[mask] = rng.choice(statuses, size=mask.sum(), p=probs)

df = pd.DataFrame({
    "channel": channels,
    "region": regions,
    "status": order_status,
    "revenue": revenue.round(2),
})

print(f"Dataset: {len(df):,} rows  |  columns: {list(df.columns)}")
df.head()

---
## Part 1 — Single-column groupby with a single aggregation

In [ ]:
mean_by_channel = (
    df.groupby("channel")["revenue"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)
print(mean_by_channel)

In [ ]:
# REVIEW QUESTION 1
#
# The code above uses .mean() to summarise revenue by channel.
# Revenue data is typically right-skewed — a small number of very large orders
# can pull the mean well above the value that represents most transactions.
#
# Before deciding whether mean is the right choice here, what would you check?
# Write your answer below. Consider: what output would convince you to switch to median?
#
# YOUR ANSWER:
# ...

In [ ]:
# Check the distribution shape before committing to the mean.
for ch in df["channel"].unique():
    sub = df.loc[df["channel"] == ch, "revenue"]
    skew = sub.skew()
    ratio = sub.mean() / sub.median()
    print(f"{ch:<12}  skew={skew:+.2f}  mean/median ratio={ratio:.2f}")

In [ ]:
# REVIEW QUESTION 2
#
# Look at the mean/median ratio for each channel.
# A ratio well above 1.0 means the mean is inflated by high-value outliers.
#
# Given what you see here, which channels (if any) would you report using median
# rather than mean in a stakeholder summary? Would you use different statistics
# for different channels, or the same one across all channels for consistency?
#
# YOUR ANSWER:
# ...

---
## Part 2 — Multiple aggregations

In [ ]:
channel_summary = (
    df.groupby("channel")["revenue"]
    .agg(count="count", mean="mean", median="median", std="std")
    .round(2)
)
print(channel_summary)

In [ ]:
# REVIEW QUESTION 3
#
# Look at the enterprise row. Its standard deviation is large — possibly
# larger than its mean. What does that tell you about the enterprise segment,
# and what would be your next analytical step?
#
# Consider: would you treat enterprise as a single homogeneous group in a model?
# What additional columns might you use to sub-segment it further?
#
# YOUR ANSWER:
# ...

---
## Part 3 — Multi-column groupby and silent NaN omission

In [ ]:
# Median revenue by channel and region — default dropna=True
pivot_default = (
    df.groupby(["channel", "region"])["revenue"]
    .median()
    .unstack()
    .round(2)
)
print("With dropna=True (default):")
print(pivot_default)
print()

# Same groupby with dropna=False — surfaces null-region rows
pivot_with_nan = (
    df.groupby(["channel", "region"], dropna=False)["revenue"]
    .median()
    .unstack()
    .round(2)
)
print("With dropna=False (NaN region surfaced):")
print(pivot_with_nan)

In [ ]:
# REVIEW QUESTION 4
#
# The two tables above differ only by dropna=True vs dropna=False.
# In the default table, some rows are silently excluded.
#
# How many rows are excluded? Run a quick count to find out.
# Then answer: should those rows be excluded, imputed, or kept as a separate group?
# What business question might help you decide?
#
# YOUR ANSWER:
# ...

In [ ]:
# Quick count of null-region rows — fill this in
null_region_count = df["region"].isna().sum()
print(f"Rows with null region: {null_region_count:,} ({null_region_count/len(df)*100:.1f}% of dataset)")

---
## Part 4 — Small-group problem in multi-column segmentation

In [ ]:
detailed = (
    df.groupby(["channel", "status"])["revenue"]
    .agg(count="count", median="median", std="std")
    .round(2)
    .reset_index()
)
print(detailed.to_string(index=False))

In [ ]:
# REVIEW QUESTION 5
#
# Find the rows in `detailed` where count is below 30.
# For those groups, evaluate: is the reported median a reliable finding?
# What would you do with these groups before presenting results to a stakeholder?
#
# YOUR ANSWER:
# ...

In [ ]:
# Identify underpopulated groups
small_groups = detailed[detailed["count"] < 30]
print(f"Groups with fewer than 30 observations ({len(small_groups)} found):")
print(small_groups.to_string(index=False))

---
## Part 5 — Which categorical features drive revenue variation?

In [ ]:
cat_cols = ["channel", "region", "status"]

for col in cat_cols:
    group_means = df.groupby(col)["revenue"].mean()
    spread = group_means.max() - group_means.min()
    print(f"{col:<12}  range of group means: ${spread:.2f}")

In [ ]:
# REVIEW QUESTION 6
#
# The code above ranks features by the range of their group means.
# 'channel' likely shows the largest spread; 'region' likely shows the smallest.
#
# Does a small range-of-means prove that 'region' is not worth encoding in a model?
# Describe one scenario where a feature with a small group-mean range could still
# be an important predictor.
#
# Also: the loop uses .mean() again. Given what you found in Part 1, does that
# affect the ranking? How would you modify the loop to be more robust?
#
# YOUR ANSWER:
# ...

---
## Part 6 — Bar chart for group comparison

In [ ]:
channel_median = (
    df.groupby("channel")["revenue"]
    .median()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 4))
channel_median.plot(kind="bar", ax=ax, color="#6272a4", edgecolor="white")
ax.set_title("Median revenue by channel")
ax.set_ylabel("Revenue ($)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# REVIEW QUESTION 7
#
# The bar chart shows median revenue by channel.
# A colleague suggests overlaying error bars using ±1 standard deviation.
#
# Evaluate this suggestion: is std the right spread measure to overlay on a median?
# What would be a more appropriate measure of spread around the median,
# and why does the choice matter when communicating to a non-technical stakeholder?
#
# YOUR ANSWER:
# ...

---
## Part 7 — Putting it together: segment profile

This final section builds a full channel profile. Review the code, then answer the question below it.

In [ ]:
profile = (
    df.groupby("channel")["revenue"]
    .agg(
        count="count",
        median="median",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        mean="mean",
        std="std",
    )
    .round(2)
)

profile["iqr"] = (profile["p75"] - profile["p25"]).round(2)
profile["mean_median_ratio"] = (profile["mean"] / profile["median"]).round(2)

print(profile.to_string())

In [ ]:
# REVIEW QUESTION 8 — synthesis
#
# You are preparing EDA findings for the team building a revenue prediction model.
# Based on everything you've reviewed in this notebook, write 3–5 bullet points
# that summarise what the segmentation analysis reveals and what modelling decisions
# it implies. For each point, state the finding and the downstream action.
#
# Example format:
#   - Finding: [what the data shows]
#     Action: [what this means for feature engineering or model design]
#
# YOUR ANSWER:
# ...